# PDF Load Test 5000 - DS (Data Scientist) v4

Run this notebook in a Colab session logged in as **test2@openmined.org**.

Run cells in order, coordinating with the DO notebook.

## Setup

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
BRANCH = "feature/support-aisi-pdf-download"
REPO = "https://github.com/OpenMined/syft-client.git"

!pip install -q \
  "syft-bg @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-bg" \
  "syft-job @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-job" \
  "syft-dataset @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-datasets" \
  "syft-permissions @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-permissions" \
  "syft-perm @ git+{REPO}@{BRANCH}#subdirectory=packages/syft-perm" \
  "syft-client @ git+{REPO}@{BRANCH}"

import syft_client as sc
print(f"syft_client version: {sc.__version__}")

In [ ]:
from pathlib import Path
import shutil
import time
import json

DO_EMAIL = "test1@openmined.org"
DS_EMAIL = "test2@openmined.org"
DATASET_NAME = "PDF_Test_5000"
JOB_NAME = "pdf_analysis_5000"

---
## DS Login

In [ ]:
ds_client = sc.login_ds(email=DS_EMAIL)
print(f"DS: {ds_client.email}")
print(f"SyftBox: {ds_client.syftbox_folder}")

---
## Add Peer

**Wait for DO to create dataset first**, then run this cell.

In [ ]:
ds_client.add_peer(DO_EMAIL)
ds_client.sync()
print(f"Peer request sent to {DO_EMAIL}")

# Debug: verify outbox was created
conn = ds_client.peer_manager.connection_router.connection_for_send_message()
conn.ds_outbox_folder_id_cache = {}
outbox_id = conn._get_outbox_folder_id_as_ds(DO_EMAIL)
print(f"Outbox folder ID: {outbox_id}")

---
## Submit Job

**Wait for DO to approve peer first**, then run these cells.

In [ ]:
# Sync to get updated peer info after DO approval
ds_client.sync()
ds_client.load_peers()
print(f"Peers: {[p.email for p in ds_client.peers]}")

# Verify outbox
conn = ds_client.peer_manager.connection_router.connection_for_send_message()
conn.ds_outbox_folder_id_cache = {}
outbox_id = conn._get_outbox_folder_id_as_ds(DO_EMAIL)
print(f"Outbox folder ID: {outbox_id}")
assert outbox_id is not None, "Outbox not found - peer approval may not be complete"

In [ ]:
# Cleanup existing job (allows re-runs)
# Check both possible paths (with and without submitter email subdir)
for job_dir in [
    ds_client.syftbox_folder / DO_EMAIL / "app_data" / "job" / JOB_NAME,
    ds_client.syftbox_folder / DO_EMAIL / "app_data" / "job" / DS_EMAIL / JOB_NAME,
]:
    if job_dir.exists():
        shutil.rmtree(job_dir)
        print(f"Cleaned up: {job_dir}")

In [ ]:
# Create job code
job_code = f'''import syft_client as sc
import os, json
from pathlib import Path

dataset_path = sc.resolve_path("syft://private/syft_datasets/{DATASET_NAME}")
dataset_dir = Path(dataset_path)

pdf_files = sorted(dataset_dir.glob("*.pdf"))
print(f"Found {{len(pdf_files)}} PDFs")

total_size = sum(len(f.read_bytes()) for f in pdf_files)

summary = {{
    "total_files": len(pdf_files),
    "total_size_mb": round(total_size / (1024 * 1024), 2),
}}
print(f"Results: {{summary}}")

os.makedirs("outputs", exist_ok=True)
with open("outputs/analysis.json", "w") as f:
    json.dump(summary, f)
'''

job_path = Path("/content/main.py")
job_path.write_text(job_code)
print("Job code written")

In [ ]:
# Submit job
ds_client.submit_python_job(
    user=DO_EMAIL,
    code_path=str(job_path),
    job_name=JOB_NAME
)
print(f"Job '{JOB_NAME}' submitted to {DO_EMAIL}")

# Verify message was uploaded to outbox on GDrive
conn = ds_client.peer_manager.connection_router.connection_for_send_message()
outbox_id = conn._get_outbox_folder_id_as_ds(DO_EMAIL)
print(f"\nOutbox folder ID: {outbox_id}")
if outbox_id:
    files_in_outbox = conn.get_file_metadatas_from_folder(outbox_id)
    print(f"Files in outbox: {len(files_in_outbox)}")
    for f in files_in_outbox:
        print(f"  {f['name']} (id={f['id']})")

---
## Get Results

**Wait for DO to run the job first**, then run this cell.

In [ ]:
ds_client.sync()
time.sleep(2)
ds_client.sync()

print("DS jobs:")
for job in ds_client.jobs:
    print(f"  {job.name} | status={job.status}")
    if job.name == JOB_NAME and hasattr(job, 'output_paths') and job.output_paths:
        for p in job.output_paths:
            if Path(p).exists() and "analysis.json" in str(p):
                print(f"\n  RESULTS: {json.loads(Path(p).read_text())}")

In [ ]:
# Debug: inspect job directory if results not showing
for base in [
    ds_client.syftbox_folder / DO_EMAIL / "app_data" / "job" / JOB_NAME,
    ds_client.syftbox_folder / DO_EMAIL / "app_data" / "job" / DS_EMAIL / JOB_NAME,
]:
    if base.exists():
        print(f"\nJob dir: {base}")
        for p in sorted(base.rglob("*")):
            if p.is_file():
                print(f"  {p.relative_to(base)}")
        print(f"  'done' marker: {(base / 'done').exists()}")